> ## SOLUTION / ANSWER KEY &mdash; Lab 3.9
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-09-attention-heatmap.ipynb`](../lab-09-attention-heatmap.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 3.9 &mdash; Real Attention from a Real Model

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 2 &middot; Module 3 &mdash; Why Transformers?**

### What you'll do
- Run a real model and ask it for its attention weights
- Extract one head's attention matrix over the real tokens
- Plot it as a heatmap and read which tokens attend to which

> **How this lab works (near-real):** these labs run **real Hugging Face Transformers** locally on CPU. Read the **Concept**, fill the real `___` blanks in **Build it** (real tokenizer / model / decoding calls), **Run it for real** to see the **actual model output**, note **What to notice**, then finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is real model output you can read. The genuine maths (attention, positional encoding, cosine) you still compute **by hand** &mdash; that is real mechanics, not a stub.

> **Models:** small, CPU-friendly models from the HF hub &mdash; `distilbert-base-uncased` (tokenizer / fill-mask), `sentence-transformers/all-MiniLM-L6-v2` (embeddings), `prajjwal1/bert-tiny` (attention), `distilgpt2` (generation). First use downloads the weights (needs network), then they are cached. The hosted "GPT API" path uses `ChatGroq` (`GROQ_API_KEY` in `.env`).

**Reference:** [Module 3 slides &mdash; Self-attention (Q/K/V)](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 3 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("USE_TF", "0")                 # these labs are torch-only; skip the TF backend
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # mute TensorFlow's C++ INFO/WARNING startup noise
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY etc. (used by the text-gen lab)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-03-09")
os.makedirs(WORK, exist_ok=True)
print("WORK:", WORK)
print("Real Hugging Face models load from the hub on first use (one-time download, then cached).")

## Concept
Attention is **interpretable**: for each token the model records where it looked. We load the real
**prajjwal1/bert-tiny**, ask for `output_attentions=True`, run a sentence through it, and pull out one
layer/head's `seq x seq` weight matrix &mdash; the *actual* attention the model computed, over its
*actual* subword tokens. Then we plot it.

> The heatmap needs `matplotlib` (already in the lab venv).

## Build it
Turn on attention output and extract one head's matrix.

In [ ]:
import torch, numpy as np
from transformers import AutoTokenizer, AutoModel

def load_attn_model():
    name = "prajjwal1/bert-tiny"
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModel.from_pretrained(name, attn_implementation="eager", output_attentions=True)
    model.eval()
    return tok, model

def real_attention(sentence, tok, model, layer=-1, head=0):
    enc = tok(sentence, return_tensors="pt")
    with torch.no_grad(): out = model(**enc)
    tokens = tok.convert_ids_to_tokens(enc['input_ids'][0])
    A = out.attentions[layer][0, head].numpy()
    return tokens, A

## Run it for real
Run a real sentence and plot its real attention.

In [ ]:
try:
    tok, model = load_attn_model()
    tokens, A = real_attention("the cat sat on the mat", tok, model)
    print("tokens:", tokens)
    print("attention matrix shape:", A.shape, "| row sums:", np.round(A.sum(axis=1), 2))
    import matplotlib.pyplot as plt
    plt.figure(figsize=(5, 4))
    plt.imshow(A, cmap="viridis")
    plt.xticks(range(len(tokens)), tokens, rotation=45, ha="right"); plt.yticks(range(len(tokens)), tokens)
    plt.title("real attention (bert-tiny, last layer, head 0)"); plt.colorbar(); plt.tight_layout()
    plt.savefig(WORK + "/real_attention.png", dpi=90); plt.show()
    print("saved:", WORK + "/real_attention.png")
except Exception as e:
    print("(If a ___ is still unfilled, fill it above. On first run the model downloads")
    print(" from the Hugging Face hub -- that needs network; re-run once it finishes.)")
    print("  reason:", type(e).__name__, "--", e)

## What to notice
- The tokens include `[CLS]` and `[SEP]` &mdash; the model's real inputs, not just the words you typed.
- Each **row still sums to 1** (it is a softmax) &mdash; the same maths you built by hand in Labs 3.3 and 3.8.
- Many tokens attend heavily to `[CLS]`/`[SEP]` &mdash; a real, well-documented behaviour. Different heads (`head=1,2,...`) and layers show different patterns.

## Your turn (open task &mdash; no grader)
Change the sentence and the `head` / `layer` arguments. Find a head whose pattern looks
**meaningful** (e.g. a word attending to a related word) and one that looks like a "junk" / `[SEP]`
head. A "good" answer: you can point to one cell of the heatmap and say what it means in plain English.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
try:
    import numpy as np
    tokens, A = real_attention("the quick brown fox", tok, model, layer=-1, head=1)   # try head 1
    print("tokens:", tokens)
    print("row sums:", np.round(A.sum(axis=1), 2))
    print("each token's most-attended token (head 1):")
    for i, t in enumerate(tokens):
        print(f"   {t:8s} -> {tokens[int(A[i].argmax())]}")
except Exception as e:
    print("(If a ___ is still unfilled, fill it above. On first run the model downloads")
    print(" from the Hugging Face hub -- that needs network; re-run once it finishes.)")
    print("  reason:", type(e).__name__, "--", e)

---
### Key takeaway
Attention maps are how researchers peek inside transformers. You just read them off a real model -- the by-hand maths from Labs 3.3/3.8 made concrete.

[&#8592; All Module 3 labs](./index.html) &nbsp;&middot;&nbsp; [Module 3 slides](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>